# Per-book FontDiffusion cache — CacThanhTruyen2 only

Generate `prepared/<book>/fd_cache/<book>/U+XXXX.png` for the tier-3 chars of each book, using the book's own style image.

**Book**: CacThanhTruyen2 (2,495 chars).

**Hardware**: GPU T4 x2, ~1.7 s/char → **~1-1.5 h**.

**Output**: `/kaggle/working/fd_cache_CacThanhTruyen2.zip` — download from Kaggle's Output panel.

**Style**: each book uses `kaggle_diffusion/style_references/<book>.png` (matches scan style of that specific book — better than universal style).

In [ ]:
!nvidia-smi || echo '⚠️  No GPU. Settings → Accelerator → GPU T4 x2.'

# Pin only what font_diffusion requires (NOT huggingface_hub — Kaggle's transformers
# needs a newer hub than font_diffusion/pyproject.toml's 0.36.0; pinning hub down
# breaks `from huggingface_hub import is_offline_mode` in transformers).
%pip install -q torch==2.9.1 torchvision==0.24.1 \
    --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

%pip install -q \
    diffusers==0.36.0 \
    accelerate==1.12.0 \
    safetensors==0.7.0 \
    einops==0.8.1 \
    kornia==0.8.2 \
    info-nce-pytorch==0.1.4 \
    fonttools==4.61.1 \
    pygame==2.6.1 \
    scikit-image==0.26.0 \
    scipy==1.17.0 \
    opencv-python 2>&1 | tail -3

print('\n⚠️  If torch was upgraded/downgraded, RESTART the kernel before next cell.\n   (Runtime → Restart, then re-run this cell — should be a no-op.)\n')

import torch, diffusers, huggingface_hub
print(f'PyTorch       {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
print(f'diffusers     {diffusers.__version__}')
print(f'huggingface_hub {huggingface_hub.__version__}  (kaggle default — must be transformers-compatible)')
assert diffusers.__version__ == '0.36.0', f'need diffusers==0.36.0, got {diffusers.__version__} — restart kernel'


In [ ]:
import os, sys, shutil
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/kaggle/working/GanNhanOCR')

if PROJECT_ROOT.exists():
    !cd {PROJECT_ROOT} && git pull --ff-only && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
assert (PROJECT_ROOT / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf').exists(), 'submodule clone failed'
print(f'✓ Repo at {PROJECT_ROOT}')

In [ ]:
# Download production weights (HF root, not the undertrained FST/checkpoint_step_1500)
from huggingface_hub import hf_hub_download

FD_HF = 'dzungpham/font-diffusion-weights'
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'PROD'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for fn in ['unet.safetensors', 'style_encoder.safetensors', 'content_encoder.safetensors']:
    dst = CKPT_DIR / fn
    if not dst.exists():
        cached = hf_hub_download(repo_id=FD_HF, filename=fn)
        shutil.copy2(cached, dst)
    sz = dst.stat().st_size / 1024 / 1024
    print(f'  ✓ {fn}  ({sz:.1f} MB)')

print(f'\n✓ Production weights ready')

In [ ]:
from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

print('Loading FontDiffusion (~30 s)...')
generator = FontDiffusionGenerator(
    ckpt_dir=str(CKPT_DIR),
    phase1_ckpt_dir=str(CKPT_DIR),
    font_path=str(PROJECT_ROOT / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf'),
    cache_dir=str(PROJECT_ROOT / 'kaggle_diffusion' / '_book_cache'),
    batch_size=8,  # T4 can handle 8 fine
)
generator._load_pipeline()
generator.pipe.guidance_scale = 2.0  # thinner strokes — matches sanity test defaults
print(f'✓ Loaded  |  guidance_scale=2.0  |  device={generator.device}')

## Sanity check (5 chars, ~5 s)

If this prints near-noise stats (std<30), the model is broken — STOP. If std=60-100, proceed.

In [ ]:
from PIL import Image
import numpy as np
from IPython.display import display

style_check = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / 'CacThanhTruyen2.png')
generator.generate(['一','二','三','人','月'], style_check, style_name='_sanity')
for c in ['一','二','三','人','月']:
    p = Path(generator.cache_dir) / '_sanity' / f'U+{ord(c):04X}.png'
    if p.exists():
        a = np.array(Image.open(p).convert('L'))
        ok = '✓' if a.std() > 30 else '✗ NOISE'
        print(f'  {c}  std={a.std():.0f}  ink={(a<128).sum():4d}  {ok}')
        display(Image.open(p).resize((96, 96)))
shutil.rmtree(Path(generator.cache_dir) / '_sanity', ignore_errors=True)

## Generate per-book caches

For each book, reads `kaggle_diffusion/exports/chars_<book>.txt`, generates each char using that book's `style_references/<book>.png`, applies erosion=2 to thin strokes, zips the output as `/kaggle/working/fd_cache_<book>.zip`.

Resume-safe: if a char's PNG already exists in the local cache, generation is skipped.

In [ ]:
import time
import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

BOOKS = ['CacThanhTruyen2']  # single book — 2,495 chars, ~1-1.5h on T4
ERODE_ITERS = 2
EXPORTS_DIR = PROJECT_ROOT / 'kaggle_diffusion' / 'exports'
STYLES_DIR = PROJECT_ROOT / 'kaggle_diffusion' / 'style_references'
WORK_OUT = Path('/kaggle/working/per_book_fd_cache')
WORK_OUT.mkdir(parents=True, exist_ok=True)


def erode_one(p: Path):
    arr = np.array(Image.open(p).convert('L'))
    if ERODE_ITERS > 0:
        arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=ERODE_ITERS)
    Image.fromarray(arr).save(p)


start_total = time.time()
for book in BOOKS:
    chars_file = EXPORTS_DIR / f'chars_{book}.txt'
    style_path = STYLES_DIR / f'{book}.png'
    assert chars_file.exists(), f'missing {chars_file}'
    assert style_path.exists(), f'missing {style_path}'

    chars = chars_file.read_text(encoding='utf-8').splitlines()
    chars = [c.strip() for c in chars if c.strip()]
    book_out = WORK_OUT / book / book  # nested so step 3 globs U+*.png recursively
    book_out.mkdir(parents=True, exist_ok=True)

    # Skip already-done
    todo = [c for c in chars if not (book_out / f'U+{ord(c):04X}.png').exists()]
    print(f'\n=== {book}: {len(chars)} chars ({len(todo)} to generate) ===')
    if not todo:
        continue

    # Aim cache_dir at this book's per-book sub-dir
    generator.cache_dir = str(WORK_OUT / book)
    t0 = time.time()

    # Process in chunks of 200 for progress / resilience
    CHUNK = 200
    for i in range(0, len(todo), CHUNK):
        batch = todo[i:i + CHUNK]
        try:
            generator.generate(batch, str(style_path), style_name=book)
        except Exception as e:
            print(f'  ⚠️  chunk {i}-{i+len(batch)}: {type(e).__name__}: {e}')
        # Erode whatever was just produced
        for c in batch:
            p = book_out / f'U+{ord(c):04X}.png'
            if p.exists():
                erode_one(p)
        if i + CHUNK < len(todo):
            torch.cuda.empty_cache()

    print(f'  ✓ {book}: {(time.time()-t0)/60:.1f} min')

    # Zip per-book output
    zip_path = Path(f'/kaggle/working/fd_cache_{book}.zip')
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path).replace('.zip', ''), 'zip', root_dir=str(WORK_OUT / book))
    print(f'  ✓ ZIP: {zip_path} ({zip_path.stat().st_size/1024/1024:.1f} MB)')

print(f'\n✓ All books done in {(time.time()-start_total)/60:.1f} min')

## Download instructions

After cell 6 finishes, the output panel of this Kaggle notebook will show the ZIP file:

```
/kaggle/working/fd_cache_CacThanhTruyen2.zip
```

Download them, then on your Mac:

```sh
cd /Users/truongmdn/TruongMDN/ThS/DoAn/GanNhanOCR
unzip -o ~/Downloads/fd_cache_CacThanhTruyen2.zip -d prepared/CacThanhTruyen2/fd_cache/

# Verify
echo "CacThanhTruyen2: $(find prepared/CacThanhTruyen2/fd_cache -name 'U+*.png' | wc -l) PNGs"
```

Then re-run step 3 + 4:

```sh
./run_pipeline.sh --config config/pipeline_cac.yaml --step 3
./run_pipeline.sh --config config/pipeline_cac.yaml --step 4
```

Step 3 will skip generation (cache already populated) and just rank/label. Step 4 builds the final dataset.